In [ ]:
from openai import OpenAI

client = OpenAI(api_key="YOUR_API_KEY")

In [5]:
import os
import json
import pandas as pd
import time
import random
from datetime import datetime, timedelta
import faiss
import numpy as np
import json
import re
from sentence_transformers import SentenceTransformer

In [20]:
sample_farmer_data = [
    {
        "farmer_id": "FARMER001",
        "full_name": "Nguyễn Văn An",
        "farm_name": "Ruộng An Phát",
        "area_ha": 2.5,
        "planting_date": "2025-09-20",
        "days_since_planting": 73,                   
        "rice_variety": "ST25",
        "location": {
            "province": "An Giang",
            "district": "Chợ Mới",
            "gps": {
                "lat": 10.51234,
                "lon": 105.43210
            }
        },
        "iot_data": {
            "temperature": 29.5,
            "humidity": 83,
            "soil_moisture": 65,      
            "soil_ph": 5.8,
            "water_level": 10,         
            "lux": 48200,
            "wind": 2.8,
            "wind_avg": 2.4,
            "detected_disease_name": "blast",
            "disease_confidence": 0.97
        },
        "summary_3d": {  
            "2025-12-02": {  
                "temperature": 29.5, "humidity": 83, "soil_moisture": 65, "soil_ph": 5.8,
                "water_level": 10, "lux": 48200, "wind": 2.8, "wind_avg": 2.4, "rain_probability": 20
            },
            "2025-12-03": { 
                "temperature": 30.2, "humidity": 78, "soil_moisture": 62, "soil_ph": 5.7,
                "water_level": 8, "lux": 52000, "wind": 3.1, "wind_avg": 2.7, "rain_probability": 10
            },
            "2025-12-04": {  
                "temperature": 31.0, "humidity": 75, "soil_moisture": 58, "soil_ph": 5.7,
                "water_level": 5, "lux": 55000, "wind": 3.5, "wind_avg": 3.0, "rain_probability": 5
            }
        },
        "expect_plan": {
            "main_recommendation": "THÁO BỚT NƯỚC",
            "execution_time": "2025-12-02T03:45:00Z",   
            "immediate_command": "THỰC HIỆN THÁO NƯỚC để mực nước còn 3-5 cm",
            "reason": "Mực nước hiện tại 10 cm quá cao trong khi lúa đã 73 ngày tuổi (giai đoạn làm đòng - trổ bông), dự báo 3 ngày tới nắng nóng, không mưa đáng kể. Cần tháo bớt nước để rễ phát triển tốt và chuẩn bị cho giai đoạn trổ đều.",
            "water_amount_detail": "Tháo dần để mực nước giảm xuống còn 3-5 cm trong vòng 24-48 giờ tới. Sau đó giữ khô nhẹ mặt ruộng (nứt chân chim) nếu không có mưa.",
            "three_day_plan": {
                "today": "Tháo nước mạnh đến khi mực nước còn khoảng 5 cm vào cuối ngày",
                "tomorrow": "Tiếp tục tháo nhẹ nếu mực nước vẫn >4 cm, mục tiêu đạt 3-4 cm",
                "day_after_tomorrow": "Giữ khô mặt ruộng (đất nứt nhẹ), chỉ Tháo lại nếu đất quá khô (<50% độ ẩm)"
            },
            "current_assessment": "Lúa ST25 đang ở cuối giai đoạn làm đòng (73 ngày tuổi). Mực nước hiện tại 10 cm là quá sâu, độ ẩm đất 65% vẫn đủ nhưng cần giảm nước để kích thích rễ ăn sâu và chuẩn bị trổ bông đều.",
            "water_command": "DRAIN_TO_3_5_CM"
        }
    },
    {
        "farmer_id": "FARMER004",
        "full_name": "Huỳnh Thị Kim Liên",
        "farm_name": "Ruộng Liên Kim",
        "area_ha": 2.0,
        "planting_date": "2025-08-25",
        "days_since_planting": 99,
        "rice_variety": "Đài Thơm 8",
        "location": {
            "province": "Tiền Giang",
            "district": "Châu Thành",
            "gps": {
                "lat": 10.40789,
                "lon": 106.27856
            }
        },
        "iot_data": {
            "temperature": 30.1,
            "humidity": 72,
            "soil_moisture": 48,
            "soil_ph": 6.0,
            "water_level": 0,          
            "lux": 58000,
            "wind": 4.1,
            "wind_avg": 3.6,
            "detected_disease_name": "none",
            "disease_confidence": 0.0
        },
        "summary_3d": {  
            "2025-12-02": { "temperature": 30.1, "humidity": 72, "soil_moisture": 48, "soil_ph": 6.0, "water_level": 0, "lux": 58000, "wind": 4.1, "wind_avg": 3.6, "rain_probability": 5 },
            "2025-12-03": { "temperature": 31.5, "humidity": 68, "soil_moisture": 42, "soil_ph": 6.1, "water_level": 0, "lux": 62000, "wind": 4.5, "wind_avg": 4.0, "rain_probability": 0 },
            "2025-12-04": { "temperature": 32.0, "humidity": 65, "soil_moisture": 38, "soil_ph": 6.2, "water_level": 0, "lux": 65000, "wind": 5.0, "wind_avg": 4.3, "rain_probability": 0 }
        },
        "expect_plan": {
            "main_recommendation": "THÁO BỚT NƯỚC",
            "execution_time": "2025-12-02T00:00:00Z",
            "immediate_command": "NGỪNG TƯỚI NƯỚC hoàn toàn, để ruộng khô tự nhiên",
            "reason": "Lúa Đài Thơm 8 đã 99 ngày tuổi, đang chín sáp ~85-90%, hạt trên gié đã vàng. Dự báo 3 ngày tới nắng nóng mạnh, không mưa → thời điểm lý tưởng để se mặt ruộng giúp hạt chắc, giảm lép, dễ thu hoạch.",
            "water_amount_detail": "Giữ ruộng khô hoàn toàn từ hôm nay đến ngày thu hoạch (dự kiến 7-8 ngày nữa).",
            "three_day_plan": {
                "today": "Đảm bảo không còn nước đọng, để đất nứt chân chim",
                "tomorrow": "Tiếp tục se ruộng, kiểm tra độ chín thực tế",
                "day_after_tomorrow": "Ruộng khô ráo, chuẩn bị máy gặt đập liên hợp"
            },
            "current_assessment": "Lúa chín đều, bông đẹp, không bệnh. Chỉ cần se ruộng đúng kỹ thuật là đạt năng suất cao và chất lượng gạo tốt.",
            "water_command": "FULL_DRAIN"
        }
    },
    {
    "farmer_id": "FARMER011",
    "full_name": "Lê Văn Hùng",
    "farm_name": "Ruộng Hùng Lúa",
    "area_ha": 1.8,
    "planting_date": "2025-10-20",
    "days_since_planting": 43,
    "rice_variety": "ST25",
    "location": {
        "province": "Đồng Tháp",
        "district": "Cao Lãnh",
        "gps": {
            "lat": 10.48921,
            "lon": 105.65789
        }
    },
    "iot_data": {
        "temperature": 30.8,
        "humidity": 79,
        "soil_moisture": 52,
        "soil_ph": 5.9,
        "water_level": 3,
        "lux": 51200,
        "wind": 3.2,
        "wind_avg": 2.9,
        "detected_disease_name": "none",
        "disease_confidence": 0.12
    },
    "summary_3d": {
        "2025-12-02": { "temperature": 30.8, "humidity": 79, "soil_moisture": 52, "soil_ph": 5.9, "water_level": 3, "lux": 51200, "wind": 3.2, "wind_avg": 2.9, "rain_probability": 5 },
        "2025-12-03": { "temperature": 31.5, "humidity": 75, "soil_moisture": 48, "soil_ph": 5.9, "water_level": 1, "lux": 53800, "wind": 3.5, "wind_avg": 3.1, "rain_probability": 0 },
        "2025-12-04": { "temperature": 32.1, "humidity": 72, "soil_moisture": 44, "soil_ph": 5.8, "water_level": 0, "lux": 56500, "wind": 4.0, "wind_avg": 3.4, "rain_probability": 0 }
    },
    "expect_plan": {
        "main_recommendation": "THÁO BỚT NƯỚC",
        "execution_time": "2025-12-02T05:30:00Z",
        "immediate_command": "Tháo NƯỚC NGAY để mực nước đạt 6–8 cm",
        "reason": "Lúa 43 ngày tuổi đang đẻ nhánh rộ, mực nước chỉ còn 3 cm và dự báo 3 ngày tới nắng nóng không mưa. Nguy cơ giảm nhánh hữu hiệu rất cao nếu để khô thêm.",
        "water_amount_detail": "Tháo mạnh trong sáng nay đạt 7–8 cm, sau đó giữ ổn định 5–8 cm suốt tuần này.",
        "three_day_plan": {
            "today": "Tháo đạt 8 cm trước 11h sáng",
            "tomorrow": "Duy trì 6–8 cm, bơm bổ sung nếu dưới 5 cm",
            "day_after_tomorrow": "Giữ mức 5–7 cm"
        },
        "current_assessment": "Giai đoạn đẻ nhánh mạnh của ST25 cần giữ nước liên tục 5–8 cm. Hiện tại đất đã bắt đầu khô, cần bơm ngay lập tức.",
        "water_command": "PUMP_TO_8_CM"
    }
},
    {
    "farmer_id": "FARMER012",
    "full_name": "Huỳnh Thị Mai",
    "farm_name": "Ruộng Mai Vàng",
    "area_ha": 3.2,
    "planting_date": "2025-09-15",
    "days_since_planting": 78,
    "rice_variety": "ST25",
    "location": {
        "province": "Kiên Giang",
        "district": "Giồ Rách",
        "gps": { "lat": 10.12345, "lon": 105.12345 }
    },
    "iot_data": {
        "temperature": 29.2,
        "humidity": 85,
        "soil_moisture": 61,
        "soil_ph": 5.7,
        "water_level": 11,
        "lux": 45800,
        "wind": 2.1,
        "wind_avg": 1.8,
        "detected_disease_name": "none",
        "disease_confidence": 0.05
    },
    "summary_3d": {
        "2025-12-02": { "temperature": 29.2, "humidity": 85, "soil_moisture": 61, "soil_ph": 5.7, "water_level": 11, "lux": 45800, "wind": 2.1, "wind_avg": 1.8, "rain_probability": 25 },
        "2025-12-03": { "temperature": 30.0, "humidity": 80, "soil_moisture": 58, "soil_ph": 5.7, "water_level": 9, "lux": 49500, "wind": 2.5, "wind_avg": 2.2, "rain_probability": 15 },
        "2025-12-04": { "temperature": 30.8, "humidity": 76, "soil_moisture": 55, "soil_ph": 5.6, "water_level": 7, "lux": 52800, "wind": 3.0, "wind_avg": 2.6, "rain_probability": 10 }
    },
    "expect_plan": {
        "main_recommendation": "THÁO BỚT NƯỚC",
        "execution_time": "2025-12-02T04:00:00Z",
        "immediate_command": "THÁO NGAY để mực nước còn 3–4 cm trong 36 giờ tới",
        "reason": "Lúa 78 ngày tuổi đang làm đòng cuối – chuẩn bị trổ (dự kiến 82–85 ngày). Mực nước 11 cm quá sâu, không tốt cho trổ đều và dễ gây nghẹt rễ.",
        "water_amount_detail": "Tháo mạnh hôm nay còn 5 cm, ngày mai còn 3–4 cm, sau đó giữ khô nhẹ mặt ruộng.",
        "three_day_plan": {
            "today": "Tháo mạnh đạt còn 5 cm cuối ngày",
            "tomorrow": "Tiếp tục tháo còn 3–4 cm",
            "day_after_tomorrow": "Giữ 2–4 cm, để se mặt nếu nắng"
        },
        "current_assessment": "Cần giảm nước ngay để kích thích trổ bông tập trung và đều đặn cho giống ST25.",
        "water_command": "DRAIN_TO_3_4_CM"
    }
},
    {
    "farmer_id": "FARMER013",
    "full_name": "Phạm Văn Tèo",
    "farm_name": "Ruộng Tèo ST",
    "area_ha": 2.0,
    "planting_date": "2025-09-10",
    "days_since_planting": 83,
    "rice_variety": "ST25",
    "location": {
        "province": "An Giang",
        "district": "Thoại Sơn",
        "gps": { "lat": 10.34567, "lon": 105.23456 }
    },
    "iot_data": {
        "temperature": 28.5,
        "humidity": 88,
        "soil_moisture": 58,
        "soil_ph": 5.6,
        "water_level": 2,
        "lux": 42100,
        "wind": 1.8,
        "wind_avg": 1.5,
        "detected_disease_name": "none",
        "disease_confidence": 0.08
    },
    "summary_3d": {
        "2025-12-02": { "temperature": 28.5, "humidity": 88, "soil_moisture": 58, "soil_ph": 5.6, "water_level": 2, "lux": 42100, "wind": 1.8, "wind_avg": 1.5, "rain_probability": 85 },
        "2025-12-03": { "temperature": 27.8, "humidity": 92, "soil_moisture": 72, "soil_ph": 5.5, "water_level": 18, "lux": 28500, "wind": 4.2, "wind_avg": 3.8, "rain_probability": 95 },
        "2025-12-04": { "temperature": 28.2, "humidity": 90, "soil_moisture": 75, "soil_ph": 5.5, "water_level": 22, "lux": 31200, "wind": 3.9, "wind_avg": 3.5, "rain_probability": 80 }
    },
    "expect_plan": {
        "main_recommendation": "TƯỚI THÊM NƯỚC",
        "execution_time": "2025-12-03T02:00:00Z",
        "immediate_command": "MỞ HẾT CỐNG XẢ TRÀN ngay lập tức, chỉ giữ lại 4–6 cm sau mưa",
        "reason": "Lúa đang trổ bông (83 ngày), cực kỳ mẫn cảm ngập úng gây lép hạt. Dự báo mưa rất to 2–3 ngày tới, phải xả trước để tránh nước dâng cao.",
        "water_amount_detail": "Xả tràn hôm nay còn 2–3 cm, sau các cơn mưa chỉ giữ lại tối đa 5–6 cm.",
        "three_day_plan": {
            "today": "Mở hết cống xả, giữ tối đa 3 cm",
            "tomorrow": "Sau mưa to, điều chỉnh giữ 5 cm",
            "day_after_tomorrow": "Duy trì 4–6 cm ổn định"
        },
        "current_assessment": "Giai đoạn trổ bông cần nước nông và ổn định, tuyệt đối không để ngập sâu.",
        "water_command": "EMERGENCY_OVERFLOW_DRAIN"
    }
},
    {
    "farmer_id": "FARMER014",
    "full_name": "Trần Quốc Bảo",
    "farm_name": "Ruộng Bảo An",
    "area_ha": 4.0,
    "planting_date": "2025-08-28",
    "days_since_planting": 96,
    "rice_variety": "ST25",
    "location": {
        "province": "Sóc Trăng",
        "district": "Mỹ Xuyên",
        "gps": { "lat": 9.67890, "lon": 105.89012 }
    },
    "iot_data": {
        "temperature": 31.2,
        "humidity": 74,
        "soil_moisture": 68,
        "soil_ph": 6.0,
        "water_level": 7,
        "lux": 54200,
        "wind": 3.6,
        "wind_avg": 3.2,
        "detected_disease_name": "none",
        "disease_confidence": 0.03
    },
    "summary_3d": {
        "2025-12-02": { "temperature": 31.2, "humidity": 74, "soil_moisture": 68, "soil_ph": 6.0, "water_level": 7, "lux": 54200, "wind": 3.6, "wind_avg": 3.2, "rain_probability": 5 },
        "2025-12-03": { "temperature": 32.0, "humidity": 70, "soil_moisture": 62, "soil_ph": 6.0, "water_level": 4, "lux": 57800, "wind": 4.1, "wind_avg": 3.7, "rain_probability": 0 },
        "2025-12-04": { "temperature": 32.5, "humidity": 68, "soil_moisture": 55, "soil_ph": 6.1, "water_level": 1, "lux": 59500, "wind": 4.5, "wind_avg": 4.0, "rain_probability": 0 }
    },
    "expect_plan": {
        "main_recommendation": "THÁO BỚT NƯỚC",
        "execution_time": "2025-12-02T06:00:00Z",
        "immediate_command": "THÁO CẠN hoàn toàn trong 4 ngày tới",
        "reason": "Lúa 96 ngày tuổi đã chín sáp >70%, cần tháo cạn để hạt chắc, giảm đổ ngã và thuận lợi thu hoạch bằng máy.",
        "water_amount_detail": "Hôm nay còn 3–4 cm → ngày mai cạn → ngày kia khô mặt ruộng hoàn toàn.",
        "three_day_plan": {
            "today": "Tháo mạnh còn 3 cm",
            "tomorrow": "Tháo cạn, để đất se mặt",
            "day_after_tomorrow": "Giữ khô hoàn toàn, nứt chân chim"
        },
        "current_assessment": "Giai đoạn chín, cần rút nước hoàn toàn để đạt năng suất và chất lượng cao nhất.",
        "water_command": "FULL_DRAIN_FOR_HARVEST"
    }
},
    {
    "farmer_id": "FARMER015",
    "full_name": "Võ Thị Kim Ngân",
    "farm_name": "Ruộng Ngân Phát",
    "area_ha": 1.5,
    "planting_date": "2025-10-10",
    "days_since_planting": 53,
    "rice_variety": "ST25",
    "location": {
        "province": "Cần Thơ",
        "district": "Ô Môn",
        "gps": { "lat": 10.07890, "lon": 105.67890 }
    },
    "iot_data": {
        "temperature": 30.5,
        "humidity": 81,
        "soil_moisture": 74,
        "soil_ph": 5.8,
        "water_level": 13,
        "lux": 49800,
        "wind": 2.5,
        "wind_avg": 2.2,
        "detected_disease_name": "none",
        "disease_confidence": 0.10
    },
    "summary_3d": {
        "2025-12-02": { "temperature": 30.5, "humidity": 81, "soil_moisture": 74, "soil_ph": 5.8, "water_level": 13, "lux": 49800, "wind": 2.5, "wind_avg": 2.2, "rain_probability": 10 },
        "2025-12-03": { "temperature": 31.3, "humidity": 77, "soil_moisture": 70, "soil_ph": 5.8, "water_level": 11, "lux": 52500, "wind": 3.0, "wind_avg": 2.7, "rain_probability": 5 },
        "2025-12-04": { "temperature": 32.0, "humidity": 73, "soil_moisture": 65, "soil_ph": 5.9, "water_level": 9, "lux": 55200, "wind": 3.4, "wind_avg": 3.1, "rain_probability": 0 }
    },
    "expect_plan": {
        "main_recommendation": "TƯỚI THÊM NƯỚC",
        "execution_time": "2025-12-02T07:00:00Z",
        "immediate_command": "THÁO CẠN để bắt đầu phơi ruộng 8–10 ngày",
        "reason": "Lúa 53 ngày tuổi đã đủ nhánh (18–22 nhánh/bụi), cần phơi ruộng để kiểm soát đẻ nhánh vô hiệu, giúp cây cứng chắc và chống đổ.",
        "water_amount_detail": "Tháo cạn hoàn toàn trong 3 ngày, để đất nứt chân chim nhẹ.",
        "three_day_plan": {
            "today": "Tháo mạnh còn 4–5 cm cuối ngày",
            "tomorrow": "Tháo cạn, chỉ còn nước lòng ruộng",
            "day_after_tomorrow": "Giữ khô mặt ruộng, đất bắt đầu nứt"
        },
        "current_assessment": "Giai đoạn cuối đẻ nhánh → chuyển sang phơi ruộng là tối ưu cho ST25 vụ Đông Xuân.",
        "water_command": "START_DRY_FIELD_SUNNING"
    }
},
    {
    "farmer_id": "FARMER016",
    "full_name": "Ngô Văn Tâm",
    "farm_name": "Ruộng Tâm Lộc",
    "area_ha": 2.7,
    "planting_date": "2025-11-28",
    "days_since_planting": 4,
    "rice_variety": "ST25",
    "location": {
        "province": "Hậu Giang",
        "district": "Vị Thủy",
        "gps": { "lat": 9.89012, "lon": 105.56789 }
    },
    "iot_data": {
        "temperature": 29.8,
        "humidity": 84,
        "soil_moisture": 38,
        "soil_ph": 6.1,
        "water_level": 0,
        "lux": 48500,
        "wind": 2.9,
        "wind_avg": 2.5,
        "detected_disease_name": "none",
        "disease_confidence": 0.01
    },
    "summary_3d": {
        "2025-12-02": { "temperature": 29.8, "humidity": 84, "soil_moisture": 38, "soil_ph": 6.1, "water_level": 0, "lux": 48500, "wind": 2.9, "wind_avg": 2.5, "rain_probability": 0 },
        "2025-12-03": { "temperature": 30.6, "humidity": 80, "soil_moisture": 34, "soil_ph": 6.1, "water_level": 0, "lux": 51500, "wind": 3.3, "wind_avg": 2.9, "rain_probability": 0 },
        "2025-12-04": { "temperature": 31.2, "humidity": 76, "soil_moisture": 31, "soil_ph": 6.2, "water_level": 0, "lux": 54000, "wind": 3.8, "wind_avg": 3.3, "rain_probability": 5 }
    },
    "expect_plan": {
        "main_recommendation": "GIỮ NGUYÊN MỰC NƯỚC",
        "execution_time": "2025-12-02T08:00:00Z",
        "immediate_command": "BƠM NƯỚC liếp 1–3 cm để giữ ẩm cho mạ mới",
        "reason": "Vừa sạ 4 ngày, đất đang khô quá sẽ làm hạt khó nảy mầm đều. Chỉ cần giữ ẩm bề mặt, không ngập sâu.",
        "water_amount_detail": "Bơm liếp 2 cm hôm nay, giữ ẩm nhẹ 3–5 ngày tới rồi mới giữ nước 3–5 cm khi mạ 2 lá.",
        "three_day_plan": {
            "today": "Bơm liếp 2–3 cm, giữ ẩm mặt ruộng",
            "tomorrow": "Duy trì ẩm nhẹ 1–3 cm",
            "day_after_tomorrow": "Giữ ẩm, chuẩn bị chuyển sang giữ nước khi mạ bén rễ"
        },
        "current_assessment": "Giai đoạn sau sạ cần giữ ẩm đều, không để khô và cũng không ngập sâu.",
        "water_command": "PUMP_LIGHT_2_3_CM_FOR_GERMINATION"
    }
},
    {
    "farmer_id": "FARMER017",
    "full_name": "Đỗ Văn Minh",
    "farm_name": "Ruộng Minh Đức",
    "area_ha": 2.1,
    "planting_date": "2025-10-02",
    "days_since_planting": 61,
    "rice_variety": "ST25",
    "location": {
        "province": "Tiền Giang",
        "district": "Châu Thành",
        "gps": {
            "lat": 10.41234,
            "lon": 106.29876
        }
    },
    "iot_data": {
        "temperature": 28.9,
        "humidity": 87,
        "soil_moisture": 70,
        "soil_ph": 5.7,
        "water_level": 8,
        "lux": 39800,
        "wind": 4.1,
        "wind_avg": 3.6,
        "detected_disease_name": "none",
        "disease_confidence": 0.07
    },
    "summary_3d": {
        "2025-12-02": {
            "temperature": 28.9, "humidity": 87, "soil_moisture": 70, "soil_ph": 5.7,
            "water_level": 8, "lux": 39800, "wind": 4.1, "wind_avg": 3.6, "rain_probability": 90
        },
        "2025-12-03": {
            "temperature": 27.5, "humidity": 94, "soil_moisture": 82, "soil_ph": 5.6,
            "water_level": 20, "lux": 21200, "wind": 6.8, "wind_avg": 5.9, "rain_probability": 95
        },
        "2025-12-04": {
            "temperature": 28.1, "humidity": 91, "soil_moisture": 79, "soil_ph": 5.6,
            "water_level": 16, "lux": 28500, "wind": 5.5, "wind_avg": 4.8, "rain_probability": 85
        }
    },
    "expect_plan": {
        "main_recommendation": "THÁO BỚT NƯỚC",
        "execution_time": "2025-12-04T01:30:00Z",
        "immediate_command": "MỞ TỐI ĐA CỐNG XẢ TRÀN ngay trong đêm để chỉ còn 4–5 cm trước khi mưa to",
        "reason": "Lúa 61 ngày tuổi đang làm đòng mạnh, cần nước 5–7 cm ổn định. Tuy nhiên dự báo mưa rất to 48 giờ tới có thể gây ngập sâu, làm nghẹt rễ và chậm trổ bông. Phải xả trước để có dư địa chứa nước mưa.",
        "water_amount_detail": "Xả mạnh đêm nay còn 4–5 cm → sau mưa ngày 3–4/12 giữ lại tối đa 6–7 cm, không để vượt 8 cm.",
        "three_day_plan": {
            "today": "Xả tràn khẩn cấp, đạt 4–5 cm trước 6h sáng",
            "tomorrow": "Sau mưa to, điều chỉnh giữ 6–7 cm",
            "day_after_tomorrow": "Duy trì ổn định 5–7 cm"
        },
        "current_assessment": "Giai đoạn làm đòng rất cần nước nhưng tuyệt đối không được ngập sâu khi có mưa lớn kéo dài.",
        "water_command": "PRE_RAIN_EMERGENCY_DRAIN"
    }
},
    {
    "farmer_id": "FARMER018",
    "full_name": "Bùi Thị Lan",
    "farm_name": "Ruộng Lan Hương",
    "area_ha": 3.5,
    "planting_date": "2025-09-07",
    "days_since_planting": 86,
    "rice_variety": "ST25",
    "location": {
        "province": "Long An",
        "district": "Tân Hưng",
        "gps": {
            "lat": 10.67890,
            "lon": 106.12345
        }
    },
    "iot_data": {
        "temperature": 32.8,
        "humidity": 68,
        "soil_moisture": 48,
        "soil_ph": 6.0,
        "water_level": 2,
        "lux": 61200,
        "wind": 4.8,
        "wind_avg": 4.2,
        "detected_disease_name": "none",
        "disease_confidence": 0.04
    },
    "summary_3d": {
        "2025-12-02": {
            "temperature": 32.8, "humidity": 68, "soil_moisture": 48, "soil_ph": 6.0,
            "water_level": 2, "lux": 61200, "wind": 4.8, "wind_avg": 4.2, "rain_probability": 0
        },
        "2025-12-03": {
            "temperature": 33.5, "humidity": 64, "soil_moisture": 42, "soil_ph": 6.1,
            "water_level": 0, "lux": 64500, "wind": 5.3, "wind_avg": 4.7, "rain_probability": 0
        },
        "2025-12-04": {
            "temperature": 34.4, "humidity": 61, "soil_moisture": 38, "soil_ph": 6.1,
            "water_level": 0, "lux": 66800, "wind": 5.8, "wind_avg": 5.1, "rain_probability": 0
        }
    },
    "expect_plan": {
        "main_recommendation": "THÁO BỚT NƯỚC",
        "execution_time": "2025-12-04T04:00:00Z",
        "immediate_command": "BƠM NGAY và duy trì mực nước 4–6 cm trong ít nhất 10 ngày tới",
        "reason": "Lúa 86 ngày tuổi đang trổ – chín sữa, rất cần nước nông liên tục để phấn hoa thụ phấn tốt và hạt chắc. Dự báo nắng nóng gay gắt không mưa trong nhiều ngày, đất khô rất nhanh.",
        "water_amount_detail": "Tháo đạt 5–6 cm trong hôm nay, sau đó Tháo bổ sung hàng ngày để luôn giữ 4–6 cm (không để dưới 3 cm).",
        "three_day_plan": {
            "today": "Tháo mạnh đạt 6 cm trước trưa",
            "tomorrow": "Tháo bổ sung giữ 5–6 cm",
            "day_after_tomorrow": "Tiếp tục duy trì 4–6 cm"
        },
        "current_assessment": "Giai đoạn trổ bông – chín sữa của ST25 cực kỳ nhạy cảm với thiếu nước, phải giữ nước nông ổn định trong điều kiện nắng nóng kéo dài.",
        "water_command": "PUMP_AND_MAINTAIN_5_CM"
    }
}
]

In [21]:
def _get_store_paths(store_name: str):
    """Tạo đường dẫn file động cho một kho tri thức cụ thể."""
    base_dir = r"D:\finalproject\KLTN\Backend\data\vector_store"
    index_path = os.path.join(base_dir, f"faiss_index_{store_name}.bin")
    docs_path = os.path.join(base_dir, f"documents_{store_name}.json")
    return index_path, docs_path

_stores = {}

def get_store(store_name: str):
    """
    Lấy một kho tri thức cụ thể. Tải từ cache nếu có, nếu không thì xây dựng mới.
    """
    if store_name in _stores:
        return _stores[store_name]

    index_path, docs_path = _get_store_paths(store_name)

    if os.path.exists(index_path) and os.path.exists(docs_path):
        try:
            print(f"Đang tải kho '{store_name}' từ cache...")
            index = faiss.read_index(index_path)
            with open(docs_path, 'r', encoding='utf-8') as f:
                documents = json.load(f)
            print(f"Tải thành công kho '{store_name}' với {index.ntotal} vector.")
            
            store_instance = {"index": index, "documents": documents}
            _stores[store_name] = store_instance
            return store_instance
        except Exception as e:
            print(f"Lỗi khi tải kho '{store_name}' từ cache: {e}. Sẽ xây dựng lại.")

def retrieve(store_name: str, query: str, k: int = 5) -> str:
    """Thực hiện truy vấn trên một kho tri thức chuyên biệt."""
    model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-mpnet-base-v2')
    store = get_store(store_name)
    if not store or store.get("index") is None:
        print(f"Truy vấn thất bại: Kho tri thức '{store_name}' chưa được khởi tạo.")
        return f"Lỗi: Cơ sở tri thức '{store_name}' không khả dụng."

    index = store["index"]
    documents = store["documents"]
    
    query_embedding = model.encode(
        [query],
        normalize_embeddings=True
    )
    
    try:
        _, indices = index.search(np.array(query_embedding, dtype=np.float32), k)
        retrieved_docs = [documents[i] for i in indices[0]]
        context = "\n---\n".join([doc['content'] for doc in retrieved_docs])
        
        print(f"Đã truy xuất {len(retrieved_docs)} đoạn văn bản từ kho '{store_name}' cho câu hỏi: '{query[:50]}...'")
        return context
    except Exception as e:
        print(f"Lỗi trong quá trình truy xuất từ kho '{store_name}': {e}")
        return "Lỗi: Đã xảy ra sự cố khi tìm kiếm thông tin."

In [22]:
import json
from datetime import datetime, timezone
from typing import Dict, List, Optional


def build_water_management_prompt(
    retrieved_context: str,
    farmer_info: dict,
    summary_3d: dict,
    iot_data: dict
) -> str:
    current_utc = datetime.now(timezone.utc).isoformat(timespec='seconds')[:-6] + 'Z'

    days_since_planting = farmer_info.get("days_since_planting", "không rõ")
    rice_variety = farmer_info.get("rice_variety", "không rõ")

    summary_json = json.dumps(summary_3d, ensure_ascii=False, indent=2)
    iot_json = json.dumps(iot_data, ensure_ascii=False, indent=2)

    return f"""
    Bạn là chuyên gia thủy lợi và canh tác lúa nước Việt Nam, đặc biệt hiểu rõ giống lúa ST25 và các giống chất lượng cao khác.

    **THỜI GIAN HIỆN TẠI:** {current_utc} (UTC)

    **THÔNG TIN RUỘNG LÚA:**
    - Giống lúa: {rice_variety}
    - Tuổi lúa: {days_since_planting} ngày sau sạ
    - Dữ liệu cảm biến IoT (thực tế hiện tại - ƯU TIÊN CAO NHẤT):
    {iot_json}

    **DỰ BÁO 3 NGÀY TỚI:**
    {summary_json}

    **KIẾN THỨC CHUYÊN SÂU (từ tài liệu nông nghiệp):**
    {retrieved_context}

    **NHIỆM VỤ:**
    Dựa trên dữ liệu IoT thực tế (ưu tiên tuyệt đối), kết hợp dự báo thời tiết và giai đoạn sinh trưởng của lúa, hãy đưa ra kế hoạch quản lý nước chính xác nhất cho 3 ngày tới.

    **QUY TẮC BẮT BUỘC:**
    - Ưu tiên dữ liệu cảm biến (water_level, soil_moisture) hơn dự báo
    - Chỉ chọn 1 trong 3 hành động: "TƯỚI THÊM NƯỚC", "THÁO BỚT NƯỚC", "GIỮ NGUYÊN MỰC NƯỚC"
    - Phải đưa ra mực nước mục tiêu cụ thể (cm) hoặc trạng thái đất (nứt chân chim, ướt nhẹ...)

    **ĐỊNH DẠNG TRẢ VỀ (CHỈ JSON, KHÔNG THÊM GÌ KHÁC):**
    ```json
    {{
        "main_recommendation": "TƯỚI THÊM NƯỚC / THÁO BỚT NƯỚC / GIỮ NGUYÊN MỰC NƯỚC",
        "execution_time": "{current_utc}",
        "immediate_command": "THỰC HIỆN THÁO NƯỚC để mực nước còn 3-5 cm",
        "reason": "Giải thích ngắn gọn, rõ ràng, có dẫn chứng từ IoT + dự báo",
        "water_amount_detail": "Tháo xuống còn 3-5 cm trong 24h" hoặc "Giữ mực nước 5-8 cm" hoặc "Tháo cạn để đất nứt nhẹ",
        "three_day_plan": {{
            "today": "Tháo mạnh đến 5 cm vào cuối ngày",
            "tomorrow": "Tiếp tục tháo nhẹ nếu còn >4 cm",
            "day_after_tomorrow": "Giữ khô mặt ruộng, chỉ bơm nếu đất quá khô"
        }},
        "current_assessment": "Lúa 73 ngày tuổi, đang cuối làm đòng, mực nước 10cm quá cao → cần tháo gấp",
        "water_command": "DRAIN_TO_3_5_CM" hoặc "FLOOD_TO_5_8_CM" hoặc "KEEP_CURRENT"
    }}"""

# ===============================================
# 6. HÀM TẠO KẾ HOẠCH (giống hệt NutrientAgent)
# ===============================================
def create_nutrient_plan(sample_item: dict, client) -> dict:
    data = sample_item.copy()
    expect_plan = data.pop("expect_plan", None)  

    days = data["days_since_planting"]
    query_for_retrieval = f"Quản lý tưới nước chi tiết giai đoạn"
    retrieved_context = retrieve("water_management", query_for_retrieval, k=6)

    # Build prompt giống hệt NutrientAgent
    prompt = build_water_management_prompt(
        retrieved_context=retrieved_context,
        farmer_info=data,
        summary_3d=data["summary_3d"],
        iot_data=data["iot_data"],
    )

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            response_format={"type": "json_object"},
            temperature=0.1
        )
        generated_plan = json.loads(response.choices[0].message.content)
        return {
            "generated_plan": generated_plan,
            "expect_plan": expect_plan,
            "match_execution_date": generated_plan.get("execution_date") == expect_plan.get("execution_date")
        }
    except Exception as e:
        return {"error": str(e)}

In [23]:
import time
from statistics import mean, stdev
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Khởi tạo model similarity một lần duy nhất
_sim_model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-mpnet-base-v2')

def text_similarity(text1: str, text2: str) -> float:
    if not text1 or not text2:
        return 0.0
    embeddings = _sim_model.encode([text1, text2], normalize_embeddings=True)
    sim = cosine_similarity([embeddings[0]], [embeddings[1]])[0][0]
    return float(sim)

from dateutil import parser

def extract_date(date_str: str) -> str:
    try:
        dt = parser.parse(date_str)
        return dt.date().isoformat()  
    except:
        return ""

def evaluate_water_management_agent(test_data: List[dict], client) -> dict:
    results = []
    latencies = []

    print(f"Bắt đầu đánh giá Quản lý Nước trên {len(test_data)} mẫu...\n")

    for i, item in enumerate(test_data):
        print(f"\n=== Test case {i+1}/{len(test_data)} | Farmer: {item['full_name']} ({item['rice_variety']}) ===")
        
        start = time.time()
        result = create_nutrient_plan(item, client)  # tên hàm vẫn giữ để tương thích backend
        latency = time.time() - start
        latencies.append(latency)
        
        print(result)

        if "error" in result:
            print("ERROR:", result["error"])
            continue

        gen = result["generated_plan"]
        exp = result["expect_plan"]

        if exp is None:
            print("Không có expect_plan để so sánh!")
            continue

        # === So sánh các trường chính xác tuyệt đối ===
        recommendation_match = gen.get("main_recommendation") == exp.get("main_recommendation")
        water_command_match = gen.get("water_command") == exp.get("water_command")
        gen_date = extract_date(gen.get("execution_time", ""))
        exp_date = extract_date(exp.get("execution_time", ""))
        execution_time_match = (gen_date == exp_date) if gen_date and exp_date else False


        # === So sánh nội dung bằng similarity ===
        reason_sim = text_similarity(
            gen.get("reason", "") or "",
            exp.get("reason", "") or ""
        )
        immediate_cmd_sim = text_similarity(
            gen.get("immediate_command", "") or "",
            exp.get("immediate_command", "") or ""
        )

        # === Đánh giá kế hoạch 3 ngày (so sánh từng ngày) ===
        gen_plan = gen.get("three_day_plan", {})
        exp_plan = exp.get("three_day_plan", {})
        day_similarities = []
        for day in ["today", "tomorrow", "day_after_tomorrow"]:
            g = gen_plan.get(day, "")
            e = exp_plan.get(day, "")
            day_similarities.append(text_similarity(g, e))
        three_day_avg_sim = mean(day_similarities) if day_similarities else 0.0

        # === Tổng điểm (cân bằng giữa chính xác và tương đồng ngữ nghĩa) ===
        overall_score = (
            (1.0 if recommendation_match else 0.4) +
            (1.0 if execution_time_match else 0.5) +
            reason_sim +
            immediate_cmd_sim +
            three_day_avg_sim
        ) / 5.0  

        case_result = {
            "test_id": i,
            "farmer_name": item['full_name'],
            "farm_name": item['farm_name'],
            "days_since_planting": item['days_since_planting'],
            "rice_variety": item['rice_variety'],
            "current_water_level": item['iot_data'].get('water_level'),

            # Chính xác tuyệt đối
            "recommendation_match": recommendation_match,
            "execution_time_match": execution_time_match,

            # Tương đồng ngữ nghĩa
            "reason_similarity": round(reason_sim, 3),
            "immediate_command_similarity": round(immediate_cmd_sim, 3),
            "three_day_plan_similarity": round(three_day_avg_sim, 3),

            # Tổng điểm
            "overall_score": round(overall_score, 3),
            "latency": round(latency, 3)
        }

        results.append(case_result)

        print(
            f"  Khuyến nghị: {gen.get('main_recommendation')} | "
            f"Match: {recommendation_match} | "
            f"Cmd: {water_command_match} | "
            f"Time: {execution_time_match}\n"
            f"  Reason sim: {case_result['reason_similarity']} | "
            f"Cmd sim: {case_result['immediate_command_similarity']} | "
            f"3-day sim: {case_result['three_day_plan_similarity']}\n"
            f"  Latency: {case_result['latency']}s | "
            f"Score: {case_result['overall_score']}"
        )

    # ===========================
    #   TỔNG KẾT
    # ===========================
    if not results:
        print("Không có kết quả hợp lệ nào!")
        return {"detailed": [], "summary": {}}

    avg_score = mean(r["overall_score"] for r in results)
    rec_accuracy = mean(1 if r["recommendation_match"] else 0 for r in results)
    time_accuracy = mean(1 if r["execution_time_match"] else 0 for r in results)
    avg_reason_sim = mean(r["reason_similarity"] for r in results)
    avg_cmd_sim = mean(r["immediate_command_similarity"] for r in results)
    avg_3day_sim = mean(r["three_day_plan_similarity"] for r in results)
    
    avg_latency = mean(latencies)
    std_latency = stdev(latencies) if len(latencies) > 1 else 0

    return {
        "detailed": results,
        "summary": {
            "total_successful_cases": len(results),
            "recommendation_accuracy": round(rec_accuracy, 3),
            "execution_time_accuracy": round(time_accuracy, 3),
            "average_reason_similarity": round(avg_reason_sim, 3),
            "average_immediate_command_similarity": round(avg_cmd_sim, 3),
            "average_three_day_plan_similarity": round(avg_3day_sim, 3),
            "average_overall_score": round(avg_score, 3),
            "average_latency": round(avg_latency, 3),
            "latency_std": round(std_latency, 3),
        }
    }

In [24]:
# ========================
# CHẠY ĐÁNH GIÁ
# ========================
evaluation_result = evaluate_water_management_agent(sample_farmer_data, client)

# In tóm tắt nhanh
s = evaluation_result["summary"]

Bắt đầu đánh giá Quản lý Nước trên 10 mẫu...


=== Test case 1/10 | Farmer: Nguyễn Văn An (ST25) ===
Đang tải kho 'water_management' từ cache...
Tải thành công kho 'water_management' với 104 vector.
Đã truy xuất 6 đoạn văn bản từ kho 'water_management' cho câu hỏi: 'Quản lý tưới nước chi tiết giai đoạn...'
{'generated_plan': {'main_recommendation': 'THÁO BỚT NƯỚC', 'execution_time': '2025-12-02T14:17:46Z', 'immediate_command': 'THỰC HIỆN THÁO NƯỚC để mực nước còn 5 cm', 'reason': 'Mực nước hiện tại là 10 cm, trong khi độ ẩm đất là 65% cho thấy đất đang ẩm ướt. Để đảm bảo lúa ST25 phát triển tốt trong giai đoạn cuối làm đòng, cần giảm mực nước xuống còn 5 cm để tránh ngập úng và tạo điều kiện cho rễ phát triển.', 'water_amount_detail': 'Tháo xuống còn 5 cm trong 24h', 'three_day_plan': {'today': 'Tháo mạnh đến 5 cm vào cuối ngày', 'tomorrow': 'Tiếp tục tháo nhẹ nếu còn >4 cm', 'day_after_tomorrow': 'Giữ khô mặt ruộng, chỉ bơm nếu đất quá khô'}, 'current_assessment': 'Lúa 73 ngày tuổi, đ

In [25]:
print("\n" + "="*60)
print("                TỔNG KẾT QUẢN LÝ NƯỚC")
print("="*60)
print(f"Tổng số case thành công           : {s['total_successful_cases']}")
print(f"Độ chính xác khuyến nghị chính     : {s['recommendation_accuracy']:.3f}")
print(f"Độ khớp thời gian thực thi         : {s['execution_time_accuracy']:.3f}")
print(f"Similarity lý do (reason)          : {s['average_reason_similarity']:.3f}")
print(f"Similarity lệnh tức thời           : {s['average_immediate_command_similarity']:.3f}")
print(f"Similarity kế hoạch 3 ngày         : {s['average_three_day_plan_similarity']:.3f}")
print(f"ĐIỂM TRUNG BÌNH TỔNG HỢP           : {s['average_overall_score']:.3f}")
print(f"Thời gian xử lý trung bình         : {s['average_latency']:.2f}s ± {s['latency_std']:.2f}s")
print("="*60 + "\n")

# In dòng FINAL đẹp
print(
    f"FINAL → Score: {s['average_overall_score']:.3f} | "
    f"Khuyến nghị: {s['recommendation_accuracy']:.3f} | "
    f"Time: {s['execution_time_accuracy']:.3f} | "
    f"Reason: {s['average_reason_similarity']:.3f} | "
    f"CmdSim: {s['average_immediate_command_similarity']:.3f} | "
    f"3Day: {s['average_three_day_plan_similarity']:.3f} | "
    f"Latency: {s['average_latency']:.2f}s"
)


                TỔNG KẾT QUẢN LÝ NƯỚC
Tổng số case thành công           : 10
Độ chính xác khuyến nghị chính     : 0.700
Độ khớp thời gian thực thi         : 0.700
Similarity lý do (reason)          : 0.632
Similarity lệnh tức thời           : 0.649
Similarity kế hoạch 3 ngày         : 0.664
ĐIỂM TRUNG BÌNH TỔNG HỢP           : 0.723
Thời gian xử lý trung bình         : 13.89s ± 1.21s

FINAL → Score: 0.723 | Khuyến nghị: 0.700 | Time: 0.700 | Reason: 0.632 | CmdSim: 0.649 | 3Day: 0.664 | Latency: 13.89s
